# Alfabetização e metas municipais

Análise do erro de previsão municipal, da comparação com metas e dos perfis territoriais. Defina `FIAP_RUN_DIR` com a pasta dos resultados: `metrics.json`, `protocol_lock.json` e `run_receipt.json`.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

run_dir = Path(os.environ.get('FIAP_RUN_DIR', '../reports'))
metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
lock = json.loads((run_dir / 'protocol_lock.json').read_text(encoding='utf-8'))
receipt = json.loads((run_dir / 'run_receipt.json').read_text(encoding='utf-8'))
assert metrics['run_id'] == lock['run_id'] == receipt['run_id']
assert metrics['protocol_lock_hash'] == lock['lock_hash']
assert receipt['temporal_lock_verified'] and receipt['evaluated_artifact_reload_verified']
print('Run:', metrics['run_id'], '| status:', metrics['status'])
print('Fonte:', receipt['source_fingerprint'])


## Taxa de alfabetização municipal

A tarefa considera a rede municipal, com uma observação por município e ano. MAE dá peso igual a cada município. Ridge, Random Forest e HistGradientBoosting compartilham a partição municipal da classificação; a taxa do ano anterior é a referência de persistência. As previsões são limitadas a 0–100 antes da avaliação.

In [ ]:
municipal = metrics['municipal']
print('Selecionado:', municipal['selected_model'])
display(pd.DataFrame([municipal['population']]))
display(pd.DataFrame(municipal['selection']['cv_results']))
display(pd.DataFrame([{'model':'selected', **municipal['test']}, {'model':'persistence', **municipal['baselines']['persistence']}]))


## Incerteza e comparação com metas

A banda exploratória usa os quantis 5% e 95% dos resíduos da reserva de 2024. Sua cobertura é medida no teste de 2025 e não garante a mesma cobertura em anos futuros. As metas conservam `>` ou `>=`: a igualdade atende somente ao segundo operador. Metas ausentes são tratadas separadamente.

In [ ]:
print(json.dumps(municipal['residual_band'], indent=2, ensure_ascii=False))
print(json.dumps(municipal['bootstrap'], indent=2, ensure_ascii=False))
observed = pd.DataFrame(municipal['evaluation_rows'])
display(observed.groupby('sigla_uf')[['observed','predicted']].mean().round(3))


## Projeção de 2026 e perfis territoriais

O modelo reajustado usa dados de 2024 e 2025, e a projeção de 2026 recebe contexto de 2025. Essa projeção ainda não dispõe de avaliação independente de resultados de 2026. Os cenários posteriores mantêm a taxa projetada de 2026 para comparação com as metas anuais.

K-Means resume o último ano observado, após a seleção dos modelos preditivos. Os grupos são descritivos e não representam previsões nem explicações causais.

In [ ]:
print(municipal['forecast_status'])
forecast = pd.DataFrame(municipal['forecast_rows'])
if not forecast.empty:
    display(forecast.groupby('sigla_uf')[['forecast','gap']].mean().round(3))
clusters = metrics['interpretation']['clusters']
print('Clustering:', clusters['status'])
if clusters['status'] == 'completed':
    print('k:', clusters['selected_k'], '| silhouette:', clusters['silhouette'], '| ARI:', clusters['stability_ari'])
    display(pd.DataFrame(clusters['profiles']))
print('\n'.join(metrics['limitations']))
